# Vector DB Explorer
Milvus Lite · BAAI/bge-small-en-v1.5 · 384-dim COSINE

**Schema:** each document has two fields — `service_name` and a `dense` vector built from `service_name + description`.

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

from vector_database import _get_client, parse_services, COLLECTION_NAME, SERVICES_FILE, vector_db_push_batch

# Build index if not already done
vector_db_push_batch()
client = _get_client()
print("Collection ready:", COLLECTION_NAME)

/opt/homebrew/Caskroom/miniforge/base/envs/IoT-engine/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/opt/homebrew/Caskroom/miniforge/base/envs/IoT-engine/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/opt/homebrew/Caskroom/miniforge/base/envs/IoT-engine/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/opt/homebrew/Caskroom/miniforge/base/envs/IoT-engine/lib/python3.11/site-packages/hugging

Collection ready: services


## 1 · Collection Stats

In [2]:
stats = client.get_collection_stats(COLLECTION_NAME)
print(f"Total documents indexed: {stats['row_count']}")

info = client.describe_collection(COLLECTION_NAME)
print("\nFields:")
for field in info["fields"]:
    print(f"  {field['name']:20s}  type={field['type']}")

Total documents indexed: 500

Fields:
  id                    type=5
  service_name          type=21
  dense                 type=101


## 2 · Raw Data — Service Names + Descriptions
Parsed directly from `assets/Services_description_V2.txt`.

In [3]:
import pandas as pd

with open(SERVICES_FILE, encoding="utf-8") as f:
    services = parse_services(f.read())

df = pd.DataFrame(services, columns=["service_name", "description"])
df["desc_preview"] = df["description"].str[:120] + "..."

print(f"{len(df)} services loaded from file\n")
df[["service_name", "desc_preview"]]

500 services loaded from file



,service_name,desc_preview
0,fruit and vegetable store,"A marrow is a fruit used as a vegetable, the m..."
1,produce market,An Agricultural Produce Market Committee (APMC...
2,grocery store,"A grocery store (AE), grocery shop (BE) or sim..."
3,asian grocery store,"In non-Asian countries, an Asian supermarket l..."
4,gourmet grocery store,Central Market is an American gourmet grocery ...
...,...,...
495,food court,A food court (in Asia-Pacific also called food...
496,public bathroom,Public Bathroom:A public bathroom is a facilit...
497,butcher shop,A butcher is a person who may slaughter animal...
498,helicopter tour agency,The black helicopter is a symbol of an alleged...


## 3 · All Service Names in the DB

In [4]:
PAGE_SIZE = 100
all_names = []
offset = 0

while True:
    rows = client.query(
        collection_name=COLLECTION_NAME,
        filter="id >= 0",
        output_fields=["service_name"],
        limit=PAGE_SIZE,
        offset=offset,
    )
    if not rows:
        break
    all_names.extend(r["service_name"] for r in rows)
    offset += PAGE_SIZE

db_df = pd.DataFrame(all_names, columns=["service_name"])
print(f"{len(db_df)} service names retrieved from Milvus\n")
db_df

500 service names retrieved from Milvus



,service_name
0,fruit and vegetable store
1,produce market
2,grocery store
3,asian grocery store
4,gourmet grocery store
...,...
495,food court
496,public bathroom
497,butcher shop
498,helicopter tour agency


## 4 · Semantic Search — Top-K Service Names

In [5]:
from vector_database import vector_search

queries = [
    "I need a coffee shop",
    "looking for a pharmacy",
    "I want pizza",
    "I need a doctor",
    "gym to work out",
    "somewhere to get a haircut",
    "I need to fix my car",
    "looking for a school for my kids",
]

K = 5
rows = []
for q in queries:
    results = vector_search(q, limit=K)
    rows.append({"query": q, "top_matches": " | ".join(results)})

results_df = pd.DataFrame(rows)
results_df

,query,top_matches
0,I need a coffee shop,coffee shop | convenience store | deli shop | ...
1,looking for a pharmacy,pharmacy | medical clinic | convenience store ...
2,I want pizza,pizza restaurant | delivery chinese restaurant...
3,I need a doctor,medical clinic | medical center | optometrist ...
4,gym to work out,gym | fitness center | rock climbing gym | spo...
5,somewhere to get a haircut,barber shop | hair salon | hairdresser | cosme...
6,I need to fix my car,tire shop | racing car parts store | car wash ...
7,looking for a school for my kids,school | primary school | educational institut...


## 5 · Interactive Single Query with Scores

In [6]:
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def search_with_scores(query: str, k: int = 10):
    q_vec = _model.encode([query], normalize_embeddings=True).tolist()
    hits = client.search(
        collection_name=COLLECTION_NAME,
        data=q_vec,
        anns_field="dense",
        search_params={"metric_type": "COSINE"},
        limit=k,
        output_fields=["service_name"],
    )
    rows = [
        {"rank": i + 1, "service_name": h["entity"]["service_name"], "cosine_score": round(h["distance"], 4)}
        for i, h in enumerate(hits[0])
    ]
    return pd.DataFrame(rows)

# Change this query and re-run the cell
QUERY = "I need somewhere to buy fresh vegetables"
K     = 10

print(f'Query: "{QUERY}"\n')
search_with_scores(QUERY, k=K)

/opt/homebrew/Caskroom/miniforge/base/envs/IoT-engine/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Query: "I need somewhere to buy fresh vegetables"



,rank,service_name,cosine_score
0,1,grocery store,0.7412
1,2,fresh food market,0.7156
2,3,gourmet grocery store,0.7099
3,4,farmers' market,0.6977
4,5,convenience store,0.6928
5,6,garden center,0.6709
6,7,health food store,0.6681
7,8,deli shop,0.6569
8,9,grocery delivery service,0.6548
9,10,asian grocery store,0.6429


I0625 12:52:09.434141 2549908 chttp2_transport.cc:1369] unix:/var/folders/p6/92s1vm8971s6k22vcrw2157r0000gn/T/tmpf6nep8ls_milvus_lite.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0625 12:52:09.434247 2549908 chttp2_transport.cc:1401] unix:/var/folders/p6/92s1vm8971s6k22vcrw2157r0000gn/T/tmpf6nep8ls_milvus_lite.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
